# setup

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !uv pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !uv pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !uv pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !uv pip install --no-deps --upgrade "torchao>=0.16.0"
!uv pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!uv pip install torchcodec
import torch; torch._dynamo.config.recompile_limit = 64;

In [ ]:
!uv pip install datasets

In [ ]:
from unsloth import FastModel
import torch


In [ ]:
gemma4_models = [
    # Gemma-4 instruct models:
    "unsloth/gemma-4-E2B-it",
    "unsloth/gemma-4-E4B-it",
    "unsloth/gemma-4-31B-it",
    "unsloth/gemma-4-26B-A4B-it",
    # Gemma-4 base models:
    "unsloth/gemma-4-E2B",
    "unsloth/gemma-4-E4B",
    "unsloth/gemma-4-31B",
    "unsloth/gemma-4-26B-A4B",
] # More models at https://huggingface.co/unsloth


Select the base model

In [ ]:

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B",
    dtype = None, # None for auto detection
    max_seq_length = 8192, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    #device_map = "balanced", # Uses 2x Tesla T4s

    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

# get chat template

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4-thinking",
     #chat_template = "gemma-4"
)

In [ ]:
from transformers import TextStreamer
# Helper function for inference
def do_gemma_4_inference(messages, max_new_tokens = 128):
    _ = model.generate(
        **tokenizer.apply_chat_template(
            messages,
            add_generation_prompt = True, # Must add for generation
            tokenize = True,
            return_dict = True,
            return_tensors = "pt",
        ).to("cuda"),
        max_new_tokens = max_new_tokens,
        use_cache = True,
        temperature = 1.0, top_p = 0.95, top_k = 64,
        streamer = TextStreamer(tokenizer, skip_prompt = True),
    )

In [ ]:
print(tokenizer.chat_template)

In [ ]:
messages = [{
    "role": "user",
    "content": [{ "type" : "text",
                  "text" : "The capital of Italy is" }]
}]
do_gemma_4_inference(messages, max_new_tokens = 32)

In [ ]:
print("messages:", messages)
print("\nchat template:\n", tokenizer.apply_chat_template(
            messages,
            add_generation_prompt = True, # Must add for generation
        ))

# Create SFT dataset

In [ ]:
# mix of dataset

In [ ]:
from datasets import load_dataset

In [ ]:
alpaca_dataset = load_dataset('vicgalle/alpaca-gpt4', split='train[:10000]')

In [ ]:
alpaca_dataset

In [ ]:
finetome_dataset = load_dataset("mlabonne/FineTome-100k", split = "train[:3000]")

In [ ]:
finetome_dataset

In [ ]:
everyday_convos = load_dataset("HuggingFaceTB/everyday-conversations-llama3.1-2k", split='train_sft')

In [ ]:
everyday_convos

In [ ]:
everyday_convos[0]['messages']

# formatting datasets for SFT

In [ ]:
alpaca_dataset[0]

In [ ]:
alpaca_dataset[0]['instruction']

In [ ]:
print(alpaca_dataset[0]['output'])

In [ ]:
# format alpaca to match the chat template format
def create_conversation_alpaca(example):
    return {
        "messages": [
            {"role": "user", "content": example['instruction']},
            {"role": "assistant", "content": example['output']},
        ]
    }

In [ ]:
alpaca_dataset

In [ ]:
alpaca_dataset[100]

In [ ]:
alpaca_dataset = alpaca_dataset.map(create_conversation_alpaca, batched=False)

In [ ]:
alpaca_dataset['messages'][0]

format finetome

In [ ]:
finetome_dataset[0]

In [ ]:
from unsloth.chat_templates import standardize_data_formats
finetome_dataset = standardize_data_formats(finetome_dataset)

In [ ]:
finetome_dataset[15]

# Apply chat template

## Alpaca dataset

In [ ]:
alpaca_dataset[10]

In [ ]:
def format_alpaca_dataset(example):
    convos = example['messages']
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False).removeprefix('<bos>') for convo in convos]
    return { "text" : texts, }

In [ ]:
alpaca_txt = alpaca_dataset.map(format_alpaca_dataset, batched=True)

In [ ]:
print(alpaca_txt['text'][10])

## Finetome txt dataset

In [ ]:
finetome_dataset[0]

In [ ]:
def format_finetome_dataset(example):
    convos = example['conversations']
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False).removeprefix('<bos>') for convo in convos]
    return { "text" : texts, }

In [ ]:
txt_finetome = finetome_dataset.map(format_finetome_dataset, batched=True)

In [ ]:
print(txt_finetome['text'][20])

## everyday convos

In [ ]:
everyday_convos[0]

In [ ]:
def format_everyday_convos(example):
    convos = example['messages']
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False).removeprefix('<bos>') for convo in convos]
    return { "text" : texts, }

In [ ]:
txt_everyday = everyday_convos.map(format_everyday_convos, batched=True)

In [ ]:
print(txt_everyday['text'][0])

Concatenate all the txt dataset

In [ ]:
txt_list = [alpaca_txt['text'], txt_finetome['text'], txt_everyday['text']]

In [ ]:
txt_list 

In [ ]:
from datasets import Dataset

In [ ]:
from datasets import Dataset, concatenate_datasets

combined_dataset = concatenate_datasets([
    Dataset.from_dict({"text": alpaca_txt['text']}),
    Dataset.from_dict({"text": txt_finetome['text']}),
    Dataset.from_dict({"text": txt_everyday['text']}),
])
combined_dataset

In [ ]:
combined_dataset.save_to_disk("../gemma_data/gemma_sft_dataset")

In [ ]:
combined_dataset = combined_dataset.shuffle(seed=42)

In [ ]:
print(combined_dataset[100]['text'])

In [ ]:
split_data =combined_dataset.train_test_split(test_size=0.1)

In [ ]:
train_data = split_data['train']
eval_data = split_data['test']

In [ ]:
train_data

In [ ]:
ls

# Train the model

## create the peft model

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # Should leave on always!

    r = 16,           # Larger = higher accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

In [ ]:
model.config.bos_token_id = tokenizer.bos_token_id
model.generation_config.bos_token_id = tokenizer.bos_token_id

In [ ]:
train_data

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_data,
    #eval_dataset = eval_data, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        per_device_eval_batch_size = 2,
        #eval_accumulation_steps = 4,
        #save_strategy = "steps",
        #save_total_limit = 3,
        #save_steps=60,
        #eval_strategy="steps",
        #eval_steps=20,
        warmup_steps = 5,
        num_train_epochs = 2, # Set this for 1 full training run.
        max_steps = 180,
        learning_rate = 2e-5, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

In [ ]:
trainer.train()

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4-thinking",
)
messages = [{
    "role": "user",
    "content": [{
        "type" : "text",
        "text" : "Continue the sequence: 1, 1, 2, 3, 5, 8,",
    }]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens = 64, # Increase for longer outputs!
    use_cache = True,
    # Recommended Gemma-4 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
)
tokenizer.batch_decode(outputs)

In [ ]:
messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "Why is the sky blue?",}]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 64, # Increase for longer outputs!
    use_cache = True,
    # Recommended Gemma-4 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

To save the final model as LoRA adapters, either use Huggingface's push_to_hub for an online save or save_pretrained for a local save.

[NOTE] This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model_save_path="gemma-4-finetune"

In [ ]:
if True: # Change to True to save finetune!
    model.save_pretrained_merged(model_save_path, tokenizer)

In [ ]:
if True:
    from unsloth import FastModel
    model, tokenizer = FastModel.from_pretrained(
        model_name = model_save_path, # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 2048,
        load_in_4bit = True,
    )

messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "What is Gemma-4?",}]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 128, # Increase for longer outputs!
    use_cache = True,
    # Recommended Gemma-4 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

In [ ]:
messages = [{
    "role": "user",
    "content": [{ "type" : "text",
                  "text" : "How many r are in strawberry?" }]
}]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")


from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 128, # Increase for longer outputs!
    use_cache = True,
    # Recommended Gemma-4 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)